# M5.2 Nodal Analysis of Networks

Book pointer: chapter 10, sections 10.1 (electric circuits) and 10.3 (circuits as models).

## 1. Recall

Answer on paper before running any cell.

1. State Kirchhoff's current law and voltage law. Which field equation does each come from in a steady state, and what does each assume about the wires and the space between them?
2. A resistor $R$ joins nodes $a$ and $b$. Write the four entries it adds to the conductance matrix $G$. Why does every row of the full $G$ sum to zero, and what has to be done before $G$ can be inverted?
3. A voltage source fixes $V_a - V_b$ but says nothing about its own current. Why does that force an extra unknown and an extra equation into nodal analysis?
4. State Thevenin's theorem. Which two measurements on a network give $V_{\text{th}}$ and $R_{\text{th}}$?
5. Draw a Wheatstone bridge and derive its balance condition $R_1/R_2 = R_3/R_4$.

## 2. Principle to algorithm

A circuit is a graph: nodes joined by components. Choose one node as ground, $V_0 = 0$, and take the other node potentials as unknowns. Kirchhoff's current law at node $i$ says the currents leaving through all its resistors add up to the current injected from outside:

$$\sum_{j} \frac{V_i - V_j}{R_{ij}} = I_i^{\text{inj}}.$$

Each resistor appears in exactly two of these equations, so the system can be built one component at a time. A resistor with conductance $g = 1/R$ between $a$ and $b$ adds $+g$ to $G_{aa}$ and $G_{bb}$ and $-g$ to $G_{ab}$ and $G_{ba}$. This is called stamping. A current source carrying $I$ from $a$ to $b$ through itself subtracts $I$ from the right-hand side at $a$ and adds it at $b$. The voltage law needs no equations of its own: writing every voltage as a difference of node potentials satisfies it around every loop automatically.

The full $G$ is singular, because adding a constant to every potential changes no current, and its rows sum to zero for the same reason. Deleting the ground row and column removes that freedom, and what is left is symmetric positive definite for any connected resistor network.

A voltage source does not fit the pattern, because its current is whatever the rest of the circuit demands. Modified nodal analysis adds that current as one more unknown. It enters the current law at its two nodes with $+1$ and $-1$, and one more row states the constraint $V_a - V_b = V_s$. Each voltage source adds one row and one column.

`emlib.circuits.mna` stamps resistors, current sources, voltage sources, capacitors and inductors into a dense matrix and solves it. That suits circuits with tens of nodes. `emlib.circuits.solve_network` takes any graph of conductances with some potentials fixed and assembles a sparse matrix. That suits the resistor lattices of notebooks 01 and 04, which have thousands of nodes. Both rest on the same current law at every node.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import circuits as ci

plt.rcParams["figure.figsize"] = (6, 4)
np.set_printoptions(precision=4, suppress=True, linewidth=110)

### 3.1 Stamping a conductance matrix by hand

Four nodes, node 0 is ground. Resistors: 1 Ω from 1 to 0, 2 Ω from 1 to 2, 1 Ω from 2 to 3, 2 Ω from 2 to 0, 1 Ω from 3 to 0. A current source pushes 1 A into node 1 from ground.

By hand, the reduced matrix (ground row and column removed) and right-hand side are

$$\begin{pmatrix} 1.5 & -0.5 & 0 \\ -0.5 & 2 & -1 \\ 0 & -1 & 2 \end{pmatrix}\begin{pmatrix} V_1 \\ V_2 \\ V_3 \end{pmatrix} = \begin{pmatrix} 1 \\ 0 \\ 0 \end{pmatrix}.$$

The third row gives $V_3 = V_2/2$. The second then gives $-0.5V_1 + 1.5V_2 = 0$, so $V_2 = V_1/3$. The first gives $1.5V_1 - V_1/6 = 1$, so $V_1 = 0.75$ V, $V_2 = 0.25$ V and $V_3 = 0.125$ V. The cell stamps the matrix one resistor at a time, prints it after each stamp, and checks the solution against `ci.mna`.

In [ ]:
resistors = [(1, 0, 1.0), (1, 2, 2.0), (2, 3, 1.0), (2, 0, 2.0), (3, 0, 1.0)]
n_nodes = 4
G = np.zeros((n_nodes, n_nodes))
for a, b, R in resistors:
    g = 1.0 / R
    G[a, a] += g; G[b, b] += g
    G[a, b] -= g; G[b, a] -= g
    print(f"after stamping {R} ohm between {a} and {b}:\n{G}\n")
print("row sums of the full G:", G.sum(axis=1))

I_inj = np.array([0.0, 1.0, 0.0, 0.0])
V_hand = np.zeros(n_nodes)
V_hand[1:] = np.linalg.solve(G[1:, 1:], I_inj[1:])          # ground row and column removed
V_mna, _ = ci.mna(n_nodes, resistors=resistors, current_sources=[(0, 1, 1.0)])
print("by stamping:", V_hand, "   by ci.mna:", V_mna)

### 3.2 A voltage source adds a row and a column

Replace the current source with a 3 V source from node 1 to ground. The unknowns are now $V_1, V_2, V_3$ and the source current $I_s$, defined as the current flowing through the source from its $+$ node to its $-$ node. The current law at node 1 gains $+I_s$, because $I_s$ leaves node 1 into the source. The new row states $V_1 = 3$:

$$\begin{pmatrix} 1.5 & -0.5 & 0 & 1 \\ -0.5 & 2 & -1 & 0 \\ 0 & -1 & 2 & 0 \\ 1 & 0 & 0 & 0 \end{pmatrix}\begin{pmatrix} V_1 \\ V_2 \\ V_3 \\ I_s \end{pmatrix} = \begin{pmatrix} 0 \\ 0 \\ 0 \\ 3 \end{pmatrix}.$$

The network is linear, so the answer follows from section 3.1 by scaling. One ampere into node 1 raised it to 0.75 V, so the network's input resistance is 0.75 Ω and 3 V drives 4 A into it. The potentials are four times the earlier ones, $V = (3, 1, 0.5)$, and $I_s = -4$ A. The minus sign means the current runs through the source from $-$ to $+$, which is what a source delivering power does.

In [ ]:
A = np.zeros((4, 4))
A[:3, :3] = G[1:, 1:]
A[0, 3] = A[3, 0] = 1.0                                     # node 1 is the source's + terminal
z = np.array([0.0, 0.0, 0.0, 3.0])
sol = np.linalg.solve(A, z)
print("augmented matrix:\n", A)
print("by hand: V1, V2, V3 =", sol[:3], "  I_s =", sol[3])
V_vs, I_vs = ci.mna(n_nodes, resistors=resistors, voltage_sources=[(1, 0, 3.0)])
print("ci.mna:  V =", V_vs[1:], "  I_s =", I_vs)
print("power delivered by the source -V_s I_s =", -3.0 * I_vs[0], "W;  sum of V^2/R =", sum((V_vs[a] - V_vs[b]) ** 2 / R for a, b, R in resistors), "W")

### 3.3 The Wheatstone bridge

Two voltage dividers share a source. $R_1$ over $R_2$ form the left divider, $R_3$ over $R_4$ the right, and a galvanometer $R_g$ joins the two midpoints. No current flows through the galvanometer when both midpoints sit at the same potential:

$$\frac{R_2}{R_1 + R_2} = \frac{R_4}{R_3 + R_4} \quad\Longleftrightarrow\quad \frac{R_1}{R_2} = \frac{R_3}{R_4}.$$

That is how an unknown resistor is measured: adjust a known one until the galvanometer reads zero. The condition does not depend on the source voltage or on $R_g$. Away from balance, the bridge seen from the galvanometer is a Thevenin source with $V_{\text{th}} = V_s\big(R_2/(R_1 + R_2) - R_4/(R_3 + R_4)\big)$ and $R_{\text{th}} = R_1 \parallel R_2 + R_3 \parallel R_4$, so $I_g = V_{\text{th}}/(R_{\text{th}} + R_g)$.

With $R_1 = 100$ Ω, $R_2 = 200$ Ω, $R_3 = 300$ Ω and the source at 10 V, balance is at $R_4 = 600$ Ω. The cell sweeps $R_4$ and compares the solver with the Thevenin formula.

In [ ]:
def bridge_current(R1, R2, R3, R4, Rg=100.0, Vs=10.0):
    # nodes: 0 bottom (ground), 1 top, 2 left midpoint, 3 right midpoint
    res = [(1, 2, R1), (2, 0, R2), (1, 3, R3), (3, 0, R4), (2, 3, Rg)]
    V, _ = ci.mna(4, resistors=res, voltage_sources=[(1, 0, Vs)])
    return (V[2] - V[3]) / Rg

def bridge_thevenin(R1, R2, R3, R4, Rg=100.0, Vs=10.0):
    Vth = Vs * (R2 / (R1 + R2) - R4 / (R3 + R4))
    Rth = R1 * R2 / (R1 + R2) + R3 * R4 / (R3 + R4)
    return Vth / (Rth + Rg)

R4s = np.linspace(300, 900, 121)
Ig = np.array([bridge_current(100, 200, 300, r) for r in R4s])
Ig_th = np.array([bridge_thevenin(100, 200, 300, r) for r in R4s])
k = np.argmin(np.abs(Ig))
print(f"zero crossing of the sweep at R4 = {R4s[k]:.1f} ohm (balance 600), I_g there = {Ig[k]:.1e} A")
print(f"largest difference from the Thevenin formula over the sweep: {np.abs(Ig - Ig_th).max():.1e} A")
plt.plot(R4s, 1e3 * Ig, label="ci.mna")
plt.plot(R4s[::10], 1e3 * Ig_th[::10], "o", label="Thevenin formula")
plt.axhline(0, color="grey", lw=0.5); plt.axvline(600, color="grey", ls=":")
plt.xlabel("R4 [ohm]"); plt.ylabel("galvanometer current [mA]"); plt.legend(); plt.title("Wheatstone bridge"); plt.show()

### 3.4 A Thevenin equivalent from two solves

Seen from two terminals, any linear network of sources and resistors behaves like one source $V_{\text{th}}$ in series with one resistor $R_{\text{th}}$. Two solves find them. The open-circuit voltage between the terminals is $V_{\text{th}}$. The short-circuit current $I_{\text{sc}}$ is found by placing a 0 V source across the terminals, which is how a short is written in modified nodal analysis. Then $R_{\text{th}} = V_{\text{th}}/I_{\text{sc}}$.

The network below has a 12 V source at node 1 and five resistors: 4 Ω (1 to 2), 12 Ω (2 to 0), 2 Ω (2 to 3), 6 Ω (3 to 0) and 8 Ω (1 to 3). The terminals are node 3 and ground. The test of the equivalent is to attach loads $R_L$ to the real network and compare $V_L$ with $V_{\text{th}}R_L/(R_{\text{th}} + R_L)$. Note that the 0 V source in the short-circuit solve carries current from node 3 to ground, from its $+$ to its $-$ node, so its $I_s$ is positive.

In [ ]:
net = [(1, 2, 4.0), (2, 0, 12.0), (2, 3, 2.0), (3, 0, 6.0), (1, 3, 8.0)]
src = [(1, 0, 12.0)]
V_open, _ = ci.mna(4, resistors=net, voltage_sources=src)
V_th = V_open[3]
_, I_sources = ci.mna(4, resistors=net, voltage_sources=src + [(3, 0, 0.0)])
I_sc = I_sources[1]
R_th = V_th / I_sc
print(f"V_th = {V_th:.4f} V   I_sc = {I_sc:.4f} A   R_th = {R_th:.4f} ohm")
print("   R_L     V_L (network)    V_L (Thevenin)")
R_loads = np.array([0.5, 1.0, 2.0, 3.0, 10.0, 100.0])
V_loads = []
for RL in R_loads:
    V, _ = ci.mna(4, resistors=net + [(3, 0, RL)], voltage_sources=src)
    V_loads.append(V[3])
    print(f"{RL:6.1f}   {V[3]:12.6f}   {V_th * RL / (R_th + RL):12.6f}")
V_loads = np.array(V_loads)

### 3.5 The R-2R ladder

A digital-to-analogue converter turns a binary number into a voltage. The R-2R ladder does it with only two resistor values. Ladder nodes $A_0$ to $A_3$ are joined in a chain by resistors $R$. A resistor $2R$ goes from $A_0$ to ground. Each bit $b_k$ drives node $A_k$ through a resistor $2R$ from a voltage $b_kV_{\text{ref}}$. The output is $A_3$.

Why it works. Looking from any ladder node towards the least significant end, the resistance is $2R$. At $A_0$ there are two $2R$ resistors to ground in parallel, giving $R$, and adding the series $R$ to $A_1$ makes $2R$ again, at every step. So each bit's $2R$ meets an equal $2R$ at its node, and its contribution halves at each step along the ladder towards the output. The most significant bit alone gives $V_{\text{ref}}/2$, the next $V_{\text{ref}}/4$, and by superposition

$$V_{\text{out}} = V_{\text{ref}}\sum_{k=0}^{3} b_k 2^{k-4} = V_{\text{ref}}\,\frac{\text{code}}{16}.$$

The cell solves all 16 codes.

In [ ]:
def r2r_output(code, V_ref=1.0, R=1.0, bits=4):
    # node 0 ground; ladder nodes 1..bits (node 1 is the LSB end, node bits is the output);
    # bit-source nodes bits+1 .. 2*bits
    res = [(1, 0, 2 * R)] + [(k, k + 1, R) for k in range(1, bits)] + [(1 + k, bits + 1 + k, 2 * R) for k in range(bits)]
    vs = [(bits + 1 + k, 0, V_ref * ((code >> k) & 1)) for k in range(bits)]
    V, _ = ci.mna(2 * bits + 1, resistors=res, voltage_sources=vs)
    return V[bits]

codes = np.arange(16)
V_out = np.array([r2r_output(c) for c in codes])
for c in (1, 2, 4, 8, 15):
    print(f"code {c:2d} = {c:04b}:  V_out = {V_out[c]:.6f} V_ref   code/16 = {c / 16:.6f}")
plt.step(codes, V_out, where="mid", label="R-2R ladder")
plt.plot(codes, codes / 16, "k.", label="code / 16")
plt.xlabel("input code"); plt.ylabel("V_out / V_ref"); plt.legend(); plt.title("4-bit R-2R converter"); plt.show()

### 3.6 A cube of resistors

Twelve 1 Ω resistors sit on the edges of a cube. What is the resistance between two corners? Number the corners 0 to 7 by their binary coordinates $(x, y, z)$. An edge joins two corners that differ in exactly one bit.

Opposite corners, by symmetry. Inject 1 A at corner 000 and take it out at 111. The current splits equally into the three edges leaving 000, so each carries 1/3 A. The three corners one step away are equivalent, and each sends its 1/3 A on through two edges, so the six middle edges carry 1/6 A each. The current then gathers through the three edges into 111, at 1/3 A each. Any path from 000 to 111 crosses one edge of each kind, so the voltage drop is $1/3 + 1/6 + 1/3 = 5/6$ V and $R = 5/6$ Ω. The same reasoning is harder for adjacent corners (7/12 Ω) and for the two ends of a face diagonal (3/4 Ω), and the solver does not care.

For two-terminal resistance with `solve_network`, fix one corner at 1 V and the other at 0 V. Sum the current leaving the first corner through its edges, and take $R = 1/I$.

In [ ]:
cube_edges = np.array([(a, b) for a in range(8) for b in range(a + 1, 8) if bin(a ^ b).count("1") == 1])
print("number of edges:", len(cube_edges))

def two_terminal_resistance(n_nodes, edges, G, a, b):
    fixed = np.zeros(n_nodes, dtype=bool); fixed[[a, b]] = True
    V_fixed = np.zeros(n_nodes); V_fixed[a] = 1.0
    V, I_edge = ci.solve_network(n_nodes, edges, G, fixed, V_fixed)
    I_out = I_edge[edges[:, 0] == a].sum() - I_edge[edges[:, 1] == a].sum()
    return 1.0 / I_out

G_cube = np.ones(len(cube_edges))
cube_cases = {"opposite corners (000-111)": (0, 7, 5 / 6), "adjacent corners (000-001)": (0, 1, 7 / 12), "face diagonal (000-011)": (0, 3, 3 / 4)}
for name, (a, b, exact) in cube_cases.items():
    R = two_terminal_resistance(8, cube_edges, G_cube, a, b)
    print(f"{name:28s} R = {R:.6f} ohm   exact {exact:.6f}")

### Exercise: stamp a conductance matrix

Replace the body of `stamp_conductance_matrix` with your own stamp loop. It should return the full $n\times n$ matrix including the ground row and column. The check does not compare your matrix with a hand-written one. It probes `ci.mna` instead: injecting 1 A into each node in turn gives one column of the inverse of the reduced matrix, so your reduced $G$ times those columns must give the identity.

In [ ]:
def stamp_conductance_matrix(n_nodes, resistors):
    '''Full conductance matrix of resistors given as (a, b, R).'''
    G = np.zeros((n_nodes, n_nodes))                        # replace from here with your own stamp loop
    for a, b, R in resistors:
        G[[a, b], [a, b]] += 1.0 / R
        G[a, b] -= 1.0 / R; G[b, a] -= 1.0 / R
    return G


test_circuit = [(0, 1, 3.0), (1, 2, 1.5), (2, 3, 2.0), (3, 0, 4.0), (1, 3, 5.0), (2, 4, 1.0), (4, 0, 2.5)]
n_test = 5
G_test = stamp_conductance_matrix(n_test, test_circuit)
responses = np.column_stack([ci.mna(n_test, resistors=test_circuit, current_sources=[(0, k, 1.0)])[0][1:] for k in range(1, n_test)])
print("reduced G times mna responses:\n", G_test[1:, 1:] @ responses)
assert np.allclose(G_test, G_test.T) and np.allclose(G_test.sum(axis=1), 0.0)
assert np.allclose(G_test[1:, 1:] @ responses, np.eye(n_test - 1))
print("exercise check passed")

## 4. Verify

Four checks, each against an exact result.

1. The cube: the three two-terminal resistances against 5/6, 7/12 and 3/4.
2. The R-2R ladder: all 16 outputs against code/16.
3. The bridge: zero galvanometer current at $R_1/R_2 = R_3/R_4$, for several resistor sets and source voltages.
4. Energy: in a random network driven by one source, the power the source delivers, $-V_sI_s$, equals the power dissipated, $\sum I^2R$, summed over every resistor. This holds only if every node satisfies the current law, so it tests the whole solution.

In [ ]:
for name, (a, b, exact) in cube_cases.items():
    assert abs(two_terminal_resistance(8, cube_edges, G_cube, a, b) - exact) < 1e-12, name
assert np.allclose(V_out, codes / 16, atol=1e-12)

rng = np.random.default_rng(7)
for _ in range(5):
    R1, R2, R3 = rng.uniform(10, 1000, 3)
    assert abs(bridge_current(R1, R2, R3, R2 * R3 / R1, Rg=rng.uniform(1, 500), Vs=rng.uniform(1, 50))) < 1e-14

# a random connected network: a ring of 12 nodes plus 15 random chords, a 5 V source between node 1 and ground
n_rand = 12
ring = [(i, (i + 1) % n_rand) for i in range(n_rand)]
chords = []
while len(chords) < 15:
    a, b = rng.integers(0, n_rand, 2)
    if a != b and (a, b) not in ring and (b, a) not in ring:
        chords.append((int(a), int(b)))
rand_res = [(a, b, float(rng.uniform(1, 10))) for a, b in ring + chords]
V_r, I_r = ci.mna(n_rand, resistors=rand_res, voltage_sources=[(1, 0, 5.0)])
P_source = -5.0 * I_r[0]
P_each = np.array([(V_r[a] - V_r[b]) ** 2 / R for a, b, R in rand_res])
print(f"random network: source delivers {P_source:.10f} W, resistors dissipate {P_each.sum():.10f} W")
assert np.isclose(P_source, P_each.sum(), rtol=1e-12)

order = np.argsort(P_each)[::-1]
plt.bar(range(len(P_each)), P_each[order])
plt.xlabel("resistor, sorted by power"); plt.ylabel("power [W]")
plt.title(f"random network: {len(rand_res)} resistors share {P_source:.3f} W"); plt.show()
print("all verify checks passed")

## 5. Explore and rebuild

Explore. An infinite square grid of 1 Ω resistors has two classic closed forms. The resistance between neighbouring nodes is exactly 1/2 Ω. One way to see it: inject 1 A at one node and remove it at infinity. By symmetry 1/4 A flows through the shared edge. Doing the same with current extracted at the neighbour and adding the two cases gives 1/2 A through that edge, so the drop is 1/2 V. The resistance between diagonal neighbours is $2/\pi \approx 0.6366$ Ω. Build finite $n\times n$ grids with `ci.grid_edges` and `ci.solve_network`, measure both resistances between nodes at the centre, and plot the error against $n$ for $n$ from 5 to 201. Does the error fall like $1/n$, $1/n^2$ or $1/\ln n$? Then join all the boundary nodes to one another with very large conductances, so the boundary acts as a single floating node, and repeat. Rayleigh's monotonicity law says removing resistors can only raise a resistance and shorting nodes can only lower it. Use it to explain why the free and the shorted sequences approach the infinite-grid value from opposite sides.

Rebuild. Close this notebook. On a blank page, write the stamp for a resistor and the extra row and column for a voltage source, then solve the four-node example of section 3.1 by hand. In a blank editor, write a function that takes a netlist of resistors, current sources and voltage sources and returns the node potentials and source currents. Check it against the R-2R ladder and the cube.